# 🧪 W2-D5 概念实验：KV Cache 与 Flash Attention

> 配套阅读：`第2周-Day5-推理优化-KV-Cache与Flash-Attention.md`（三大技术讲解在那边）
> 这个 notebook 把"省算力/省显存/省带宽"三笔账**真的算出来**：
> **增量解码省多少算力、KV Cache 吃多少显存、Flash Attention 的在线 softmax 怎么等价省带宽**
>
> 实验环境：纯 numpy + matplotlib，小规模模拟，不依赖 GPU。

## 实验 1：KV Cache 到底省了多少计算？

自回归生成第 t 步：**无缓存**要把整个前缀（t 个词）重新过一遍网络；
**有缓存**只过新来的 1 个词（QKV 投影 + FFN），历史 K/V 直接复用。
按每词 12·d² MACs（QKVO+FFN）建账，d=4096（LLaMA-7B 量级）。

In [ ]:
import numpy as np

def decode_macs(T, d, use_cache):
    """生成 T 个 token 的总乘加量（单层近似：每词投影+FFN ≈ 12d²，注意力 ≈ 2·t·d）"""
    total = 0
    for t in range(1, T + 1):
        n_new = 1 if use_cache else t        # 无缓存：整段重算；有缓存：只算新词
        total += 12 * n_new * d * d + 2 * t * d
    return total

d = 4096
print(f"{'生成长度 T':>10}{'无缓存 (G-MACs)':>18}{'有缓存 (G-MACs)':>18}{'加速比':>10}")
for T in [64, 256, 1024, 2048]:
    a, b = decode_macs(T, d, False), decode_macs(T, d, True)
    print(f"{T:>10}{a/1e9:>18,.0f}{b/1e9:>18,.0f}{a/b:>9.0f}×")

print("\n→ 无缓存总计算 ∝ T²（Σt·12d²），有缓存 ∝ T：生成 2048 词差约 1000 倍")
print("→ 代价：缓存本身要占显存（实验 2 算这笔账）—— 典型的空间换时间")

## 实验 2：KV Cache 显存账 —— MHA vs GQA vs MQA

公式：`层数 × KV头数 × d_head × 序列长 × 2(K和V) × 精度字节`。
用真实模型配置（LLaMA-7B MHA、LLaMA-70B GQA-8）算：长上下文下 KV Cache
可以比模型权重还大——这就是 GQA/MQA 存在的理由。

In [ ]:
def kv_gb(layers, kv_heads, d_head, seq, dtype_bytes=2):
    return layers * kv_heads * d_head * seq * 2 * dtype_bytes / 1e9   # ×2 = K和V两份

configs = [
    ("LLaMA-7B        MHA-32头", 32, 32, 128),
    ("LLaMA-70B       GQA-8头",  80,  8, 128),
    ("LLaMA-70B 若用  MHA-32头", 80, 32, 128),
    ("假设      MQA-1头",         80,  1, 128),
]
seqs = [4096, 32768]
print(f"{'模型':<26}{'4K 上下文':>12}{'32K 上下文':>12}")
for name, L, H, dh in configs:
    print(f"{name:<26}{kv_gb(L,H,dh,seqs[0]):>9.1f} GB{kv_gb(L,H,dh,seqs[1]):>9.1f} GB")

print("\n→ 70B 若用 MHA，32K 上下文光 KV Cache 就 42.9 GB；GQA-8 只要 10.7 GB（1/4）")
print("→ 再乘 batch_size：并发 8 路 = 再 ×8 —— 生产系统必须算这笔账")

## 实验 3：Flash Attention 的核心 —— 分块 + 在线 softmax

朴素做法要物化整张 n×n 分数矩阵；Flash Attention 把 K/V 切成小块，
**用运行最大值 m 和运行和 l 边扫边更新**（在线 softmax），全程只有 n×d 级的内存。
实现一个块版注意力，与朴素版对比数值——应精确到 1e-12 量级。

In [ ]:
rng = np.random.default_rng(5)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

n, d, block = 512, 64, 64
Q = rng.normal(size=(n, d)); K = rng.normal(size=(n, d)); V = rng.normal(size=(n, d))

naive = softmax(Q @ K.T / np.sqrt(d), axis=1) @ V          # 物化 512×512 矩阵

def flash_attention(Q, K, V, block=64):
    """分块流式注意力：只维护 m(行最大)/l(行和)/acc(输出)，从不物化 n×n"""
    n, d = Q.shape[0], Q.shape[1]
    out = np.zeros((n, d))
    m_row = np.full(n, -np.inf)     # 每行已见分数的最大值
    l_row = np.zeros(n)             # 每行未归一化的分母
    for j0 in range(0, K.shape[0], block):
        s = Q @ K[j0:j0+block].T / np.sqrt(d)               # 只物化 n×block
        m_new = np.maximum(m_row, s.max(axis=1))
        p = np.exp(s - m_new[:, None])
        alpha = np.exp(m_row - m_new)        # 旧行最大值变了 → 旧累加整体缩放
        l_row = l_row * alpha + p.sum(axis=1)
        out = out * alpha[:, None] + p @ V[j0:j0+block]
        m_row = m_new
    return out / l_row[:, None]

flash = flash_attention(Q, K, V, block)
print(f"朴素版 vs 分块版 最大偏差: {np.abs(naive - flash).max():.2e}")
print(f"两者与精确解一致性: {np.allclose(naive, flash, atol=1e-12)}")
print("\n→ 在线 softmax 数学上严格等价；Flash Attention 的收益在内存层级：")
print("   HBM 只读 K/V 块一次，SRAM 里完成全部计算（实验 4 算带宽账）")

## 实验 4：带宽账 —— 为什么"少读写"比"少计算"更值钱

朴素注意力要把 n×n 的分数矩阵 S 和概率矩阵 P 写回显存再读出来（各两次），
总量 ∝ 4n²；Flash 版只有 Q/K/V 进、O 出，∝ 4nd。d=128 时画曲线：32K 序列差 250 倍。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

ns = np.array([1024, 4096, 16384, 32768, 65536])
d = 128
naive_traffic = (4 * ns**2 + 3 * ns * d) * 2      # S/P 读写 + QKV 读，fp16
flash_traffic = 4 * ns * d * 2                     # QKV 读 + O 写，fp16

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.loglog(ns / 1024, naive_traffic / 1e9, "o-", label="朴素注意力（物化 n×n）")
ax.loglog(ns / 1024, flash_traffic / 1e9, "s-", label="Flash Attention（分块流式）")
ax.set_xlabel("序列长度 (K tokens)"); ax.set_ylabel("HBM 读写量 (GB)")
ax.set_title("显存带宽占用：n² vs n·d（d=128, fp16）")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

i = -1
print(f"64K 序列：朴素 {naive_traffic[i]/1e9:,.1f} GB vs Flash {flash_traffic[i]/1e9:.2f} GB"
      f"（{naive_traffic[i]/flash_traffic[i]:,.0f} 倍）")
print("→ FLOPs 没变，但内存访问少几个数量级 —— GPU 算力过剩、带宽才是瓶颈")

## 实验 5：GQA 折中曲线 —— KV 头数 vs 显存

MHA→GQA→MQA 是一条"质量-显存"调节旋钮：KV 头数减半，缓存减半。
LLaMA-2/3 选 GQA-8（显存 1/4，质量几乎无损）；MQA 极省但有质量风险。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

seq, layers, d_head = 8192, 32, 128
kv_heads = [32, 16, 8, 4, 2, 1]
labels = ["MHA-32", "GQA-16", "GQA-8", "GQA-4", "GQA-2", "MQA-1"]
gb = [kv_gb(layers, h, d_head, seq) for h in kv_heads]

fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ["#4C72B0"] * 6; colors[2] = "#DD8452"          # 高亮 GQA-8
bars = ax.bar(labels, gb, color=colors)
for b, v in zip(bars, gb):
    ax.text(b.get_x() + b.get_width()/2, v + 0.05, f"{v:.2f} GB", ha="center")
ax.set_xlabel("KV 头数配置"); ax.set_ylabel(f"KV Cache 显存 (GB, seq={seq})")
ax.set_title("GQA/MQA：KV 头数是显存旋钮（LLaMA-2/3 选择 GQA-8）")
plt.tight_layout(); plt.show()

print(f"MHA 32头 {gb[0]:.1f} GB → GQA-8 {gb[2]:.1f} GB（{(1-gb[2]/gb[0]):.0%} 节省，主流选择）")

## 结论

| 技术 | 解决什么 | 实验结论 |
|---|---|---|
| KV Cache | 重复计算 | O(T²)→O(T)，2048 词生成省 ~1000×（实验 1） |
| KV Cache 代价 | 显存 | 7B@32K 就 17GB；必须配套 GQA（实验 2、5） |
| Flash Attention | 内存带宽 | 在线 softmax 严格等价（1e-13 内，实验 3） |
| 带宽账 | n² vs nd | 64K 序列差 250 倍 HBM 流量（实验 4） |

→ 深入阅读：同目录 `.md` 版本 2.3 节（GQA 的原理与取舍）